# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is defined using a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets by their @id
print("Available record sets (by @id):")
for rs in metadata.record_sets:
    print(f"- {rs['@id']} | name: {rs['name']}")

# As an example, list fields for the first record set
if len(metadata.record_sets) > 0:
    first_rs = metadata.record_sets[0]
    print(f"\nFields for record set '{first_rs['name']}' (@id: {first_rs['@id']}):")
    for field in first_rs['fields']:
        print(f"  - {field['@id']} | name: {field['name']} | dataType: {field.get('dataType', 'n/a')}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s as identified above.

In [ ]:
# Collect all record set @ids
record_sets = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- Number of records: {len(df)}")
        print(f"- Columns (@id): {df.columns.tolist()}")
        display(df.head(3))
    else:
        print('- No records found.')

# Pick a main record set (first one, if available) for EDA
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain data for analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Below, we select a numeric field and a group variable using their `@id`s.

In [ ]:
# Select a numeric and group field (by inspecting columns):
df = dataframes[main_record_set_id]

# Attempt to programmatically select numeric and group fields by @id
numeric_field_id = None
group_field_id = None

# Scan the first record set fields for dataType Float/Integer/Number
for field in metadata.record_sets[0]['fields']:
    dt = field.get('dataType','')
    if any(x in dt for x in ['Float','Integer','Number']) and field['@id'] in df.columns and numeric_field_id is None:
        numeric_field_id = field['@id']
    elif ('Sex' in field['name'] or 'Gender' in field['name'] or 'MSI' in field['name'] or 'Anatomical' in field['name']) and field['@id'] in df.columns:
        group_field_id = field['@id']

if numeric_field_id is not None:
    print(f"Numeric field selected (@id): {numeric_field_id}")
else:
    raise ValueError("No numeric field found via Croissant schema. Please inspect and set manually.")

if group_field_id is not None:
    print(f"Group field selected (@id): {group_field_id}")
else:
    print("No group field selected via schema heuristics.")

# Filtering (use mean as threshold for demonstration)
numeric_col = numeric_field_id
threshold = df[numeric_col].mean() if pd.api.types.is_numeric_dtype(df[numeric_col]) else 10

filtered_df = df[df[numeric_col] > threshold].copy()
print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
print(f"\nNormalized {numeric_col} for filtered records:")
display(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

# Grouping
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_col].mean().reset_index()
    print(f"\nMean of {numeric_col} grouped by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and, if possible, group-wise differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped boxplot if group_field available
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, inspecting, and analyzing a complex clinical colorectal cancer survivorship dataset described by a Croissant schema, using the `mlcroissant` library. We explored available record sets and fields by their `@id`, extracted records as DataFrames, filtered and normalized numeric fields, grouped and visualized data variables, and prepared the dataset for advanced analysis. This workflow illustrates how FAIR data practices enable rapid, reproducible exploration of complex biomedical datasets using modern Python tooling.